# Django Unmanaged Models

## Introduction to Unmanaged Models
---

In this lesson, you'll learn when and how to use **unmanaged models** in Django.

Unmanaged models let you work with database tables without Django controlling their schema.

1. [What is managed = False](#What-is-managed-=-False),
    - [Managed vs Unmanaged](#Managed-vs-Unmanaged),
    - [When inspectdb Sets managed = False](#When-inspectdb-Sets-managed-=-False),
2. [When to Use Unmanaged Models](#When-to-Use-Unmanaged-Models),
    - [Legacy Database Integration](#Legacy-Database-Integration),
    - [Shared Databases](#Shared-Databases),
    - [Read-Only Access](#Read-Only-Access),
3. [Limitations of Unmanaged Models](#Limitations-of-Unmanaged-Models),
    - [What Doesn't Work](#What-Doesn't-Work),
    - [Workarounds](#Workarounds),
4. [Best Practices for Legacy Databases](#Best-Practices-for-Legacy-Databases),
    - [Project Structure](#Project-Structure),
    - [Testing Strategy](#Testing-Strategy),
5. [Switching Between Managed and Unmanaged](#Switching-Between-Managed-and-Unmanaged),
    - [Taking Over Schema Management](#Taking-Over-Schema-Management),
    - [Fake Migrations](#Fake-Migrations),
6. [🧠 EXERCISE 🧠, Work with Unmanaged Models](#🧠-EXERCISE-🧠,-Work-with-Unmanaged-Models).

<br>

## What is `managed = False`

---

### Managed vs Unmanaged

---

By default, Django **manages** your database tables:

```python
class Blog(models.Model):
    title = models.CharField(max_length=200)
    
    class Meta:
        managed = True  # This is the default!
```

<br>

**What "managed" means:**

| `managed = True` (default) | `managed = False` |
|--------------------------|------------------|
| `makemigrations` creates migrations | No migrations generated |
| `migrate` creates/alters tables | Tables must exist already |
| Django controls schema | External system controls schema |
| Full ORM features | Some limitations |

<br>

**Example: Unmanaged model**

In [ ]:
# An unmanaged model - Django won't touch this table

class Blog(models.Model):
    """Blog posts from legacy database."""
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    author_email = models.EmailField(blank=True)
    slug = models.SlugField(max_length=200, blank=True)
    content = models.TextField(blank=True)
    published_date = models.DateField()

    class Meta:
        managed = False        # Django won't create/modify this table
        db_table = 'blog_blog' # Maps to existing table in legacy_db

<br>

### When `inspectdb` Sets `managed = False`

---

`inspectdb` always generates models with `managed = False` because:

1. The tables **already exist** - no need to create them
2. Another system might **own** the schema
3. Prevents **accidental changes** to production data

<br>

```python
# Generated by inspectdb
class Employees(models.Model):
    emp_id = models.AutoField(primary_key=True)
    # ... fields ...

    class Meta:
        managed = False  # Always set by inspectdb
        db_table = 'employees'
```

<br>

## When to Use Unmanaged Models

---

### Legacy Database Integration

---

**Scenario:** You're building a new Django app that needs to read from an existing database.

<br>

```
# EXAMPLE:
┌──────────────────────────────────────────────────┐
│                  Legacy Database                  │
│  ┌───────────┐  ┌───────────┐  ┌───────────┐    │
│  │ employees │  │   orders  │  │ products  │    │
│  └───────────┘  └───────────┘  └───────────┘    │
└──────────────────────────────────────────────────┘
         │                │                │
         ▼                ▼                ▼
┌──────────────────────────────────────────────────┐
│              Django (Read-Only Access)            │
│  ┌───────────┐  ┌───────────┐  ┌───────────┐    │
│  │ Employee  │  │   Order   │  │  Product  │    │
│  │managed=F  │  │managed=F  │  │managed=F  │    │
│  └───────────┘  └───────────┘  └───────────┘    │
└──────────────────────────────────────────────────┘
```

<br>

**Why unmanaged:** The legacy system (PHP, Java, etc.) still manages the database schema.

<br>

### Shared Databases

---

**Scenario:** Multiple applications share the same database.

<br>

```
                    ┌───────────────┐
                    │   Database    │
                    │               │
                    │  ┌─────────┐  │
                    │  │ users   │  │
                    │  └─────────┘  │
                    └───────┬───────┘
                            │
           ┌────────────────┼────────────────┐
           │                │                │
           ▼                ▼                ▼
    ┌─────────────┐  ┌─────────────┐  ┌─────────────┐
    │  Django App │  │  Java App   │  │   Go API    │
    │ managed=F   │  │   (owner)   │  │ managed=F   │
    └─────────────┘  └─────────────┘  └─────────────┘
```

<br>

**Why unmanaged:** Only one application should own the schema. Others use `managed = False`.

<br>

### Read-Only Access

---

**Scenario:** You have read-only access to a data warehouse for reporting.

<br>

```python
# Data warehouse models - read-only access

class BlogStats(models.Model):
    """Aggregated blog statistics from data warehouse."""
    published_date = models.DateField(primary_key=True)
    total_posts = models.IntegerField()
    total_comments = models.IntegerField()
    avg_rating = models.DecimalField(max_digits=4, decimal_places=2)
    
    class Meta:
        managed = False
        db_table = 'daily_blog_summary'  # View in data warehouse
```

<br>

**Why unmanaged:** You don't have permission to modify the schema, only SELECT data.

<br>

## Limitations of Unmanaged Models

---

### What Doesn't Work

---

| Feature | Managed | Unmanaged |
|---------|---------|------------|
| `makemigrations` | ✅ Creates migration | ❌ No migration created |
| `migrate` | ✅ Creates/alters table | ❌ Does nothing |
| Add new field | ✅ Migration adds column | ❌ Must add column manually |
| Remove field | ✅ Migration removes column | ❌ Must remove column manually |
| Constraints in model | ✅ Applied to DB | ❌ Ignored (model only) |
| Django admin | ✅ Full support | ⚠️ Works but can't change schema |

<br>

**Key limitation: Constraints are NOT enforced at database level**

In [ ]:
class BlogReview(models.Model):
    blog = models.ForeignKey('Blog', on_delete=models.DO_NOTHING)
    rating = models.PositiveSmallIntegerField(
        validators=[MaxValueValidator(5)]  # Only validated in Django, not DB
    )
    comment = models.TextField(blank=True)
    slug = models.SlugField(unique=True)  # unique=True is ignored!

    class Meta:
        managed = False
        db_table = 'blog_reviews'
        constraints = [
            models.CheckConstraint(
                check=models.Q(rating__gte=1) & models.Q(rating__lte=5),
                name='valid_rating'  # This constraint is NOT created!
            )
        ]

<br>

**What this means:**

- `unique=True` - Only warns in Django admin, DB doesn't enforce it
- `CheckConstraint` - Never created in database
- `validators` - Only run in Django, raw SQL can bypass them

<br>

### Workarounds

---

**1. Use model validation consistently**

In [ ]:
class BlogReview(models.Model):
    blog = models.ForeignKey('Blog', on_delete=models.DO_NOTHING)
    rating = models.PositiveSmallIntegerField()
    comment = models.TextField(blank=True)

    class Meta:
        managed = False
        db_table = 'blog_reviews'

    def clean(self):
        if not (1 <= self.rating <= 5):
            raise ValidationError({'rating': 'Rating must be between 1 and 5.'})

    def save(self, *args, **kwargs):
        self.full_clean()  # Calls clean()
        super().save(*args, **kwargs)

**2. Check constraints exist in legacy DB**

Verify that critical constraints exist in the legacy database:

```sql
-- Check existing constraints in MS SQL Server
SELECT constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE table_name = 'blog_reviews';
```

**3. Request DBA to add constraints**

If constraints are missing, ask the database administrator to add them:

```sql
-- Add missing constraint to legacy table
ALTER TABLE blog_reviews
ADD CONSTRAINT valid_rating CHECK (rating >= 1 AND rating <= 5);
```

<br>

## Best Practices for Legacy Databases

---

### Project Structure

---

**Recommended structure for projects with legacy database:**

```
mysite/
├── manage.py
├── docker-compose.yml
├── mysite/                     # Project configuration
│   ├── settings.py
│   └── urls.py
├── blog/                      # Your main app (managed models)
│   ├── models.py              # New tables Django manages
│   ├── views.py
│   └── ...
├── legacy/                    # Legacy integration app
│   ├── models.py              # Unmanaged models only
│   ├── admin.py               # Admin for legacy data (read-only)
│   └── apps.py
└── backup/                     # DB backups (.bak files)
```

<br>

**Separate app for legacy models:**

In [ ]:
# blog/models_legacy.py - All unmanaged models in one place

from django.db import models


class BaseLegacyModel(models.Model):
    """Base class for all legacy models."""

    class Meta:
        abstract = True
        managed = False  # All legacy models are unmanaged


class LegacyBlog(BaseLegacyModel):
    """Legacy blog posts."""
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    author_email = models.EmailField(blank=True)
    slug = models.SlugField(max_length=200, blank=True)
    content = models.TextField(blank=True)
    published_date = models.DateField()

    class Meta(BaseLegacyModel.Meta):
        db_table = 'blog_blog'


class LegacyComment(BaseLegacyModel):
    """Legacy comments on blog posts."""
    blog = models.ForeignKey(
        LegacyBlog,
        on_delete=models.DO_NOTHING,  # Legacy - don't cascade
        db_column='blog_id',
    )
    content = models.TextField()
    created_at = models.DateTimeField()

    class Meta(BaseLegacyModel.Meta):
        db_table = 'blog_comment'

<br>

**Read-only admin for legacy models:**

In [ ]:
from django.contrib import admin
from blog.models import LegacyBlog, LegacyComment


class ReadOnlyAdminMixin:
    """Mixin to make admin read-only."""

    def has_add_permission(self, request):
        return False

    def has_change_permission(self, request, obj=None):
        return False

    def has_delete_permission(self, request, obj=None):
        return False


@admin.register(LegacyBlog)
class LegacyBlogAdmin(ReadOnlyAdminMixin, admin.ModelAdmin):
    list_display = ['id', 'title', 'author', 'published_date']
    list_filter = ['published_date']


@admin.register(LegacyComment)
class LegacyCommentAdmin(ReadOnlyAdminMixin, admin.ModelAdmin):
    list_display = ['id', 'blog', 'created_at']
    list_filter = ['blog']

<br>

### Testing Strategy

---

**Problem:** Django test database won't create unmanaged tables!

<br>

**Solution 1: Temporarily set managed = True for tests**

In [ ]:
# conftest.py (pytest) or test setup

import pytest
from django.apps import apps


@pytest.fixture(scope='session')
def django_db_setup(django_db_setup, django_db_blocker):
    """Make unmanaged models managed for test database."""
    with django_db_blocker.unblock():
        # Get all unmanaged models
        for model in apps.get_models():
            if not model._meta.managed:
                # Temporarily set managed = True
                model._meta.managed = True

<br>

**Solution 2: Use settings override**

In [ ]:
# legacy/models.py

from django.conf import settings
from django.db import models


class Employee(models.Model):
    emp_id = models.AutoField(primary_key=True)
    first_name = models.CharField(max_length=50)
    
    class Meta:
        # Use managed = True only during tests
        managed = getattr(settings, 'TESTING', False)
        db_table = 'employees'

In [ ]:
# settings.py

import sys

# Set TESTING = True when running tests
TESTING = 'pytest' in sys.modules or 'test' in sys.argv

<br>

**Solution 3: Test against real legacy DB (carefully!)**

```python
# settings_test.py

from .settings import *

# Use a copy of legacy database for testing
DATABASES['default'] = {
    'ENGINE': 'django.db.backends.mssql',
    'NAME': 'blog_test',  # Test copy!
    # ...
}
```

<br>

## Switching Between Managed and Unmanaged

---

### Taking Over Schema Management

---

Sometimes you want Django to **take over** managing a legacy table:

<br>

**Step 1: Change `managed = False` to `managed = True`**

In [ ]:
class LegacyBlog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    # ...

    class Meta:
        managed = True  # Changed from False!
        db_table = 'blog_blog'

<br>

**Step 2: Create initial migration (without applying)**

```bash
python manage.py makemigrations legacy
# legacy --> name of the app
```

This creates a migration that would CREATE the table.

<br>

**Step 3: Fake the migration (table already exists)**

```bash
python manage.py migrate legacy --fake
```

This marks the migration as applied WITHOUT running it.

<br>

### Fake Migrations

---

`--fake` tells Django "mark this migration as applied, but don't run it".

<br>

**When to use `--fake`:**

| Scenario | Command |
|----------|----------|
| Table exists, Django doesn't know | `migrate --fake` |
| Taking over legacy table | `migrate --fake` |
| Database was manually modified | `migrate --fake` |
| Starting fresh with existing DB | `migrate --fake-initial` |

<br>

**Complete workflow for taking over legacy table:**

```bash
# 1. Change managed = True in model

# 2. Create migration
python manage.py makemigrations legacy

# 3. Fake the initial migration (table exists)
python manage.py migrate legacy --fake

# 4. Now you can make changes normally
# Edit model...
python manage.py makemigrations legacy
python manage.py migrate legacy  # This one runs for real
```

<br>

**Warning signs that schema doesn't match:**

```bash
# If Django model doesn't match actual table:
>>> Blog.objects.first()
# django.db.utils.ProgrammingError: column "new_field" does not exist
```

<br>

## Working with Database Views

---

Unmanaged models can also map to **database views** (not just tables):

```sql
-- Create a view in MS SQL Server
CREATE VIEW daily_blog_summary AS
SELECT
    b.published_date,
    COUNT(DISTINCT b.id)   AS total_posts,
    COUNT(c.id)            AS total_comments,
    AVG(CAST(r.rating AS DECIMAL(4,2))) AS avg_rating
FROM blog_blog b
LEFT JOIN blog_comment  c ON c.blog_id = b.id
LEFT JOIN blog_reviews  r ON r.blog_id = b.id
GROUP BY b.published_date;
```

In [ ]:
# Map Django model to the view

class BlogStats(models.Model):
    """Aggregated blog statistics per day."""
    published_date = models.DateField(primary_key=True)
    total_posts = models.IntegerField()
    total_comments = models.IntegerField()
    avg_rating = models.DecimalField(max_digits=4, decimal_places=2)

    class Meta:
        managed = False
        db_table = 'daily_blog_summary'  # Points to the view

    def __str__(self) -> str:
        return f"{self.published_date}: {self.total_posts} posts, avg rating {self.avg_rating}"

<br>

**Usage in Django shell (`python manage.py shell`):**

```python
>>> from blog.models_legacy import LegacyBlog
>>> for post in LegacyBlog.objects.using('legacy').all():
...     print(f"{post.published_date}: {post.title} ({post.author})")
2024-01-15: Getting Started with Django (John Doe)
2024-02-10: Advanced ORM Techniques (Jane Smith)
```

<br>

**Note:** Views are typically read-only. Attempting to save will fail.

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **managed = False** | Django won't create/modify the table |
| **When to use** | Legacy DB, shared DB, read-only access |
| **Limitations** | No migrations, constraints not enforced |
| **Best practice** | Separate app for legacy models |
| **Testing** | Make managed = True during tests |
| **Taking over** | Change to managed = True + fake migration |
| **Database views** | Can map to views, read-only |
| **Validation** | Use model clean() for validation |

<br>

### 🧠 EXERCISE 🧠, Work with Unmanaged Models

---

Set up a project structure for working with legacy database:

1. Create a new Django app called `legacy`
2. Move the legacy models (from lesson 13b) to `legacy/models.py`
3. Create a `BaseLegacyModel` abstract base class with `managed = False`
4. Create a read-only admin for the legacy models
5. Test that `makemigrations` doesn't create migrations for these models
6. Query the legacy data through Django ORM

<details>
    <summary>▶️ Solution</summary>
    
**1. Create app:**

```bash
python manage.py startapp legacy
```

Add `'legacy'` to `INSTALLED_APPS` in settings.py.

**2-3. Create `legacy/models.py`:**

```python
from django.db import models


class BaseLegacyModel(models.Model):
    """Base class for all legacy models."""
    
    class Meta:
        abstract = True
        managed = False


class Department(BaseLegacyModel):
    dept_id = models.AutoField(primary_key=True)
    name = models.CharField(max_length=100, db_column='dept_name')
    code = models.CharField(unique=True, max_length=10, db_column='dept_code')
    
    class Meta(BaseLegacyModel.Meta):
        db_table = 'departments'
    
    def __str__(self):
        return f"{self.code} - {self.name}"


class Employee(BaseLegacyModel):
    emp_id = models.AutoField(primary_key=True)
    first_name = models.CharField(max_length=50)
    last_name = models.CharField(max_length=50)
    email = models.EmailField(unique=True, max_length=100)
    hire_date = models.DateField()
    salary = models.DecimalField(max_digits=10, decimal_places=2, null=True)
    department = models.ForeignKey(
        Department,
        on_delete=models.DO_NOTHING,
        null=True,
        db_column='dept_id'
    )
    is_active = models.BooleanField(default=True)
    
    class Meta(BaseLegacyModel.Meta):
        db_table = 'employees'
    
    def __str__(self):
        return f"{self.first_name} {self.last_name}"
```

**4. Create `legacy/admin.py`:**

```python
from django.contrib import admin
from .models import Department, Employee


class ReadOnlyAdminMixin:
    def has_add_permission(self, request):
        return False
    
    def has_change_permission(self, request, obj=None):
        return False
    
    def has_delete_permission(self, request, obj=None):
        return False


@admin.register(Department)
class DepartmentAdmin(ReadOnlyAdminMixin, admin.ModelAdmin):
    list_display = ['dept_id', 'name', 'code']


@admin.register(Employee)
class EmployeeAdmin(ReadOnlyAdminMixin, admin.ModelAdmin):
    list_display = ['emp_id', 'first_name', 'last_name', 'department', 'is_active']
    list_filter = ['department', 'is_active']
```

**5. Verify no migrations:**

```bash
python manage.py makemigrations legacy
# Should output: No changes detected in app 'legacy'
```

**6. Test queries:**

```python
>>> from legacy.models import Department, Employee
>>> Department.objects.all()
>>> Employee.objects.select_related('department').all()
```

</details>

<br>

### 🧠 EXERCISE 🧠, Create a Database View Model

---

Create a database view and map it to a Django model:

1. Create a SQL script that creates a view `department_stats` with:
   - `dept_name`
   - `employee_count`
   - `total_salary`
   - `avg_salary`
2. Add the script to `scripts/init-db/` and restart Docker
3. Create a `DepartmentStats` unmanaged model
4. Query the view through Django

<details>
    <summary>▶️ Solution</summary>
    
**1. Create `scripts/init-db/02-views.sql`:**

```sql
CREATE OR REPLACE VIEW department_stats AS
SELECT 
    d.dept_name,
    COUNT(e.emp_id) as employee_count,
    COALESCE(SUM(e.salary), 0) as total_salary,
    COALESCE(AVG(e.salary), 0) as avg_salary
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name;
```

**2. Restart Docker:**

```bash
docker compose down -v
docker compose up -d
```

**3. Add to `legacy/models.py`:**

```python
class DepartmentStats(BaseLegacyModel):
    """View with department statistics."""
    dept_name = models.CharField(max_length=100, primary_key=True)
    employee_count = models.IntegerField()
    total_salary = models.DecimalField(max_digits=12, decimal_places=2)
    avg_salary = models.DecimalField(max_digits=10, decimal_places=2)
    
    class Meta(BaseLegacyModel.Meta):
        db_table = 'department_stats'
    
    def __str__(self):
        return f"{self.dept_name}: {self.employee_count} employees"
```

**4. Query:**

```python
>>> from legacy.models import DepartmentStats
>>> for stats in DepartmentStats.objects.all():
...     print(f"{stats.dept_name}: {stats.employee_count} emps, ${stats.avg_salary} avg")
Engineering: 2 emps, $80000.00 avg
Marketing: 1 emps, $65000.00 avg
Human Resources: 0 emps, $0.00 avg
```

</details>

---